In [ ]:
"""
ATM Management System
----------------------
Single-file Python program simulating an ATM with:
- PIN-based login
- Balance inquiry
- Cash withdrawal (with denomination-aware dispensing simulation)
- Cash deposit
- PIN change
- Mini statement (transaction history)
- Persistent storage using JSON file (accounts.json)

Run:
    python atm_management_system.py
"""

import json
import os
import random
import string
from datetime import datetime

DATA_FILE = "accounts.json"


# --------------------------- Data Layer --------------------------- #

def load_accounts():
    """Load accounts from JSON file. Create file with a demo account if missing."""
    if not os.path.exists(DATA_FILE):
        demo_data = {
            "1234567890": {
                "name": "Demo User",
                "pin": "1234",
                "balance": 5000.0,
                "transactions": []
            }
        }
        save_accounts(demo_data)
        return demo_data

    with open(DATA_FILE, "r") as f:
        return json.load(f)


def save_accounts(accounts):
    with open(DATA_FILE, "w") as f:
        json.dump(accounts, f, indent=4)


# --------------------------- Utility Functions --------------------------- #

def generate_account_number():
    return "".join(random.choices(string.digits, k=10))


def log_transaction(account, txn_type, amount, balance_after):
    account["transactions"].append({
        "type": txn_type,
        "amount": amount,
        "balance_after": balance_after,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })
    # Keep only the last 10 transactions for a mini statement
    account["transactions"] = account["transactions"][-10:]


def clear_screen():
    os.system("cls" if os.name == "nt" else "clear")


def pause():
    input("\nPress Enter to continue...")


# --------------------------- Core ATM Operations --------------------------- #

def create_account(accounts):
    clear_screen()
    print("===== CREATE NEW ACCOUNT =====")
    name = input("Enter your name: ").strip()

    while True:
        pin = input("Set a 4-digit PIN: ").strip()
        if pin.isdigit() and len(pin) == 4:
            break
        print("Invalid PIN. It must be exactly 4 digits.")

    initial_deposit = 0.0
    while True:
        amount_str = input("Enter initial deposit amount (min 500): ").strip()
        try:
            initial_deposit = float(amount_str)
            if initial_deposit < 500:
                print("Minimum initial deposit is 500.")
                continue
            break
        except ValueError:
            print("Please enter a valid number.")

    acc_no = generate_account_number()
    while acc_no in accounts:
        acc_no = generate_account_number()

    accounts[acc_no] = {
        "name": name,
        "pin": pin,
        "balance": initial_deposit,
        "transactions": []
    }
    log_transaction(accounts[acc_no], "Account Opened", initial_deposit, initial_deposit)
    save_accounts(accounts)

    print(f"\nAccount created successfully!")
    print(f"Your Account Number is: {acc_no}")
    print("Please keep this number safe. You'll need it to log in.")
    pause()


def login(accounts):
    clear_screen()
    print("===== ATM LOGIN =====")
    acc_no = input("Enter Account Number: ").strip()

    if acc_no not in accounts:
        print("Account not found.")
        pause()
        return None

    for attempt in range(3):
        pin = input("Enter 4-digit PIN: ").strip()
        if pin == accounts[acc_no]["pin"]:
            print(f"\nWelcome, {accounts[acc_no]['name']}!")
            pause()
            return acc_no
        else:
            remaining = 2 - attempt
            if remaining > 0:
                print(f"Incorrect PIN. {remaining} attempt(s) left.")
            else:
                print("Too many incorrect attempts. Card blocked for this session.")
    pause()
    return None


def check_balance(accounts, acc_no):
    clear_screen()
    account = accounts[acc_no]
    print("===== BALANCE INQUIRY =====")
    print(f"Account Holder: {account['name']}")
    print(f"Current Balance: Rs. {account['balance']:.2f}")
    pause()


def deposit_cash(accounts, acc_no):
    clear_screen()
    account = accounts[acc_no]
    print("===== CASH DEPOSIT =====")
    try:
        amount = float(input("Enter amount to deposit: Rs. "))
        if amount <= 0:
            print("Amount must be positive.")
            pause()
            return
        if amount % 100 != 0:
            print("Please deposit in multiples of 100.")
            pause()
            return

        account["balance"] += amount
        log_transaction(account, "Deposit", amount, account["balance"])
        save_accounts(accounts)

        print(f"\nDeposit successful! New Balance: Rs. {account['balance']:.2f}")
    except ValueError:
        print("Invalid amount entered.")
    pause()


def dispense_denominations(amount):
    """Simulate note dispensing using standard Indian denominations."""
    denominations = [2000, 500, 200, 100]
    result = {}
    remaining = amount
    for note in denominations:
        count = remaining // note
        if count > 0:
            result[note] = int(count)
            remaining -= count * note
    return result


def withdraw_cash(accounts, acc_no):
    clear_screen()
    account = accounts[acc_no]
    print("===== CASH WITHDRAWAL =====")
    try:
        amount = float(input("Enter amount to withdraw: Rs. "))
        if amount <= 0:
            print("Amount must be positive.")
            pause()
            return
        if amount % 100 != 0:
            print("Please withdraw in multiples of 100.")
            pause()
            return
        if amount > account["balance"]:
            print("Insufficient balance.")
            pause()
            return
        if amount > 50000:
            print("Withdrawal limit exceeded. Max Rs. 50,000 per transaction.")
            pause()
            return

        account["balance"] -= amount
        log_transaction(account, "Withdrawal", amount, account["balance"])
        save_accounts(accounts)

        notes = dispense_denominations(amount)
        print("\nPlease collect your cash:")
        for note, count in notes.items():
            print(f"  {count} x Rs.{note} note(s)")
        print(f"\nTransaction successful! New Balance: Rs. {account['balance']:.2f}")
    except ValueError:
        print("Invalid amount entered.")
    pause()


def change_pin(accounts, acc_no):
    clear_screen()
    account = accounts[acc_no]
    print("===== CHANGE PIN =====")
    old_pin = input("Enter current PIN: ").strip()
    if old_pin != account["pin"]:
        print("Incorrect current PIN.")
        pause()
        return

    new_pin = input("Enter new 4-digit PIN: ").strip()
    if not (new_pin.isdigit() and len(new_pin) == 4):
        print("Invalid PIN format. Must be 4 digits.")
        pause()
        return

    confirm_pin = input("Confirm new PIN: ").strip()
    if new_pin != confirm_pin:
        print("PINs do not match.")
        pause()
        return

    account["pin"] = new_pin
    save_accounts(accounts)
    print("PIN changed successfully!")
    pause()


def mini_statement(accounts, acc_no):
    clear_screen()
    account = accounts[acc_no]
    print("===== MINI STATEMENT (Last 10 transactions) =====")
    if not account["transactions"]:
        print("No transactions yet.")
    else:
        print(f"{'Date/Time':<20}{'Type':<18}{'Amount':<12}{'Balance':<12}")
        print("-" * 62)
        for txn in reversed(account["transactions"]):
            print(f"{txn['timestamp']:<20}{txn['type']:<18}"
                  f"Rs.{txn['amount']:<10.2f}Rs.{txn['balance_after']:<10.2f}")
    pause()


# --------------------------- Menus --------------------------- #

def account_menu(accounts, acc_no):
    while True:
        clear_screen()
        account = accounts[acc_no]
        print(f"===== ATM MENU (Welcome, {account['name']}) =====")
        print("1. Check Balance")
        print("2. Deposit Cash")
        print("3. Withdraw Cash")
        print("4. Change PIN")
        print("5. Mini Statement")
        print("6. Logout")
        choice = input("Enter your choice (1-6): ").strip()

        if choice == "1":
            check_balance(accounts, acc_no)
        elif choice == "2":
            deposit_cash(accounts, acc_no)
        elif choice == "3":
            withdraw_cash(accounts, acc_no)
        elif choice == "4":
            change_pin(accounts, acc_no)
        elif choice == "5":
            mini_statement(accounts, acc_no)
        elif choice == "6":
            print("Logging out... Thank you for using our ATM!")
            pause()
            break
        else:
            print("Invalid choice. Try again.")
            pause()


def main_menu():
    accounts = load_accounts()
    while True:
        clear_screen()
        print("===== WELCOME TO PYTHON BANK ATM =====")
        print("1. Login")
        print("2. Create New Account")
        print("3. Exit")
        choice = input("Enter your choice (1-3): ").strip()

        if choice == "1":
            acc_no = login(accounts)
            if acc_no:
                account_menu(accounts, acc_no)
        elif choice == "2":
            create_account(accounts)
        elif choice == "3":
            print("Thank you for visiting. Goodbye!")
            break
        else:
            print("Invalid choice. Try again.")
            pause()


if __name__ == "__main__":
    main_menu()

===== WELCOME TO PYTHON BANK ATM =====
1. Login
2. Create New Account
3. Exit


Enter your choice (1-3):  2


===== CREATE NEW ACCOUNT =====


Enter your name:  khalid
Set a 4-digit PIN:  khalid


Invalid PIN. It must be exactly 4 digits.


...


Welcome, khalid!



Press Enter to continue... 


===== ATM MENU (Welcome, khalid) =====
1. Check Balance
2. Deposit Cash
3. Withdraw Cash
4. Change PIN
5. Mini Statement
6. Logout


Enter your choice (1-6):  6\


Invalid choice. Try again.


In [3]:
# hotel managenet system

import json
import os
from datetime import datetime, date

DATA_FILE = "hotel_data.json"

# Room type -> price per night
ROOM_RATES = {
    "Single": 1500,
    "Deluxe": 2800,
    "Suite": 5000
}

GST_RATE = 0.12  # 12% GST on room charges


def load_data():
    """Load hotel data (rooms, guests, bookings) from JSON file, or initialize fresh."""
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE, "r") as f:
            return json.load(f)

    # Initialize 15 rooms: 5 Single, 5 Deluxe, 5 Suite
    rooms = {}
    room_no = 101
    for rtype, count in [("Single", 5), ("Deluxe", 5), ("Suite", 5)]:
        for _ in range(count):
            rooms[str(room_no)] = {
                "type": rtype,
                "rate": ROOM_RATES[rtype],
                "is_occupied": False,
                "guest_id": None
            }
            room_no += 1

    data = {
        "rooms": rooms,
        "guests": {},        # guest_id -> guest info
        "bookings": [],       # list of booking records (history)
        "next_guest_id": 1
    }
    save_data(data)
    return data


def save_data(data):
    with open(DATA_FILE, "w") as f:
        json.dump(data, f, indent=4)


data = load_data()
print("Hotel data loaded successfully!")
print(f"Total rooms: {len(data['rooms'])}")


def today_str():
    return date.today().strftime("%Y-%m-%d")


def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def days_between(checkin_str, checkout_str):
    d1 = datetime.strptime(checkin_str, "%Y-%m-%d")
    d2 = datetime.strptime(checkout_str, "%Y-%m-%d")
    nights = (d2 - d1).days
    return max(nights, 1)  # minimum 1 night billing


def divider(char="-", length=55):
    print(char * length)


def show_all_rooms(data):
    divider("=")
    print(f"{'Room No':<10}{'Type':<10}{'Rate/Night':<12}{'Status':<12}")
    divider()
    for room_no, info in sorted(data["rooms"].items(), key=lambda x: int(x[0])):
        status = "Occupied" if info["is_occupied"] else "Available"
        print(f"{room_no:<10}{info['type']:<10}Rs.{info['rate']:<10}{status:<12}")
    divider("=")


def show_available_rooms(data, room_type=None):
    divider("=")
    print(f"{'Room No':<10}{'Type':<10}{'Rate/Night':<12}")
    divider()
    found = False
    for room_no, info in sorted(data["rooms"].items(), key=lambda x: int(x[0])):
        if not info["is_occupied"] and (room_type is None or info["type"] == room_type):
            print(f"{room_no:<10}{info['type']:<10}Rs.{info['rate']:<10}")
            found = True
    if not found:
        print("No available rooms found for this criteria.")
    divider("=")


def check_in(data):
    divider("=")
    print("GUEST CHECK-IN")
    divider()

    print("Room types available:", ", ".join(ROOM_RATES.keys()))
    room_type = input("Enter desired room type: ").strip().title()
    if room_type not in ROOM_RATES:
        print("Invalid room type.")
        return

    show_available_rooms(data, room_type)
    room_no = input("Enter room number to book: ").strip()

    if room_no not in data["rooms"]:
        print("Invalid room number.")
        return
    if data["rooms"][room_no]["is_occupied"]:
        print("Room is already occupied. Choose another.")
        return
    if data["rooms"][room_no]["type"] != room_type:
        print("Room type mismatch.")
        return

    name = input("Guest full name: ").strip()
    phone = input("Guest phone number: ").strip()
    id_proof = input("ID proof number (Aadhar/Passport): ").strip()
    checkin_date = today_str()

    guest_id = str(data["next_guest_id"])
    data["next_guest_id"] += 1

    data["guests"][guest_id] = {
        "name": name,
        "phone": phone,
        "id_proof": id_proof,
        "room_no": room_no,
        "checkin_date": checkin_date,
        "checkout_date": None,
        "active": True
    }

    data["rooms"][room_no]["is_occupied"] = True
    data["rooms"][room_no]["guest_id"] = guest_id

    save_data(data)

    print(f"\nCheck-in successful!")
    print(f"Guest ID: {guest_id}  |  Room: {room_no} ({room_type})")
    print("Please keep the Guest ID safe, you'll need it at checkout.")


def generate_bill(data, guest_id):
    guest = data["guests"][guest_id]
    room = data["rooms"][guest["room_no"]]

    nights = days_between(guest["checkin_date"], guest["checkout_date"])
    room_charge = nights * room["rate"]
    gst = round(room_charge * GST_RATE, 2)
    total = round(room_charge + gst, 2)

    divider("=")
    print("HOTEL INVOICE".center(55))
    divider("=")
    print(f"Guest Name   : {guest['name']}")
    print(f"Guest ID     : {guest_id}")
    print(f"Room No      : {guest['room_no']} ({room['type']})")
    print(f"Check-in     : {guest['checkin_date']}")
    print(f"Check-out    : {guest['checkout_date']}")
    print(f"Nights Stayed: {nights}")
    divider()
    print(f"Room Charges : Rs. {room_charge:.2f}")
    print(f"GST (12%)    : Rs. {gst:.2f}")
    divider()
    print(f"TOTAL AMOUNT : Rs. {total:.2f}")
    divider("=")

    return {
        "guest_id": guest_id,
        "guest_name": guest["name"],
        "room_no": guest["room_no"],
        "room_type": room["type"],
        "checkin_date": guest["checkin_date"],
        "checkout_date": guest["checkout_date"],
        "nights": nights,
        "room_charge": room_charge,
        "gst": gst,
        "total": total,
        "billed_on": now_str()
    }


def check_out(data):
    divider("=")
    print("GUEST CHECK-OUT")
    divider()

    guest_id = input("Enter Guest ID: ").strip()

    if guest_id not in data["guests"] or not data["guests"][guest_id]["active"]:
        print("Invalid Guest ID or guest already checked out.")
        return

    guest = data["guests"][guest_id]
    guest["checkout_date"] = today_str()

    bill = generate_bill(data, guest_id)
    data["bookings"].append(bill)

    # Free up the room
    room_no = guest["room_no"]
    data["rooms"][room_no]["is_occupied"] = False
    data["rooms"][room_no]["guest_id"] = None
    guest["active"] = False

    save_data(data)
    print("\nCheck-out complete. Room is now available.")


def current_guests(data):
    divider("=")
    print("CURRENTLY CHECKED-IN GUESTS")
    divider()
    active = {gid: g for gid, g in data["guests"].items() if g["active"]}
    if not active:
        print("No guests currently checked in.")
    else:
        print(f"{'Guest ID':<10}{'Name':<20}{'Room':<8}{'Check-in':<12}")
        divider()
        for gid, g in active.items():
            print(f"{gid:<10}{g['name']:<20}{g['room_no']:<8}{g['checkin_date']:<12}")
    divider("=")


def booking_history(data):
    divider("=")
    print("BOOKING HISTORY (All Checkouts)")
    divider()
    if not data["bookings"]:
        print("No completed bookings yet.")
    else:
        print(f"{'Guest':<18}{'Room':<8}{'Nights':<8}{'Total':<12}{'Checkout':<12}")
        divider()
        for b in data["bookings"]:
            print(f"{b['guest_name']:<18}{b['room_no']:<8}{b['nights']:<8}"
                  f"Rs.{b['total']:<10.2f}{b['checkout_date']:<12}")
    divider("=")


def revenue_report(data):
    total_revenue = sum(b["total"] for b in data["bookings"])
    total_bookings = len(data["bookings"])
    occupied = sum(1 for r in data["rooms"].values() if r["is_occupied"])
    total_rooms = len(data["rooms"])

    divider("=")
    print("HOTEL SUMMARY REPORT")
    divider()
    print(f"Total Completed Bookings : {total_bookings}")
    print(f"Total Revenue Generated  : Rs. {total_revenue:.2f}")
    print(f"Rooms Currently Occupied : {occupied}/{total_rooms}")
    print(f"Occupancy Rate           : {(occupied/total_rooms)*100:.1f}%")
    divider("=")


def main_menu():
    global data
    while True:
        divider("=")
        print("HOTEL MANAGEMENT SYSTEM".center(55))
        divider("=")
        print("1. Show All Rooms")
        print("2. Show Available Rooms")
        print("3. Guest Check-In")
        print("4. Guest Check-Out (Generate Bill)")
        print("5. View Currently Checked-In Guests")
        print("6. Booking History")
        print("7. Revenue / Summary Report")
        print("8. Exit")
        divider()

        choice = input("Enter your choice (1-8): ").strip()

        if choice == "1":
            show_all_rooms(data)
        elif choice == "2":
            show_available_rooms(data)
        elif choice == "3":
            check_in(data)
        elif choice == "4":
            check_out(data)
        elif choice == "5":
            current_guests(data)
        elif choice == "6":
            booking_history(data)
        elif choice == "7":
            revenue_report(data)
        elif choice == "8":
            print("Thank you for using the Hotel Management System. Goodbye!")
            break
        else:
            print("Invalid choice, please try again.")


# Run the system
main_menu()




Hotel data loaded successfully!
Total rooms: 15
                HOTEL MANAGEMENT SYSTEM                
1. Show All Rooms
2. Show Available Rooms
3. Guest Check-In
4. Guest Check-Out (Generate Bill)
5. View Currently Checked-In Guests
6. Booking History
7. Revenue / Summary Report
8. Exit
-------------------------------------------------------


Enter your choice (1-8):  1


Room No   Type      Rate/Night  Status      
-------------------------------------------------------
101       Single    Rs.1500      Available   
102       Single    Rs.1500      Available   
103       Single    Rs.1500      Available   
104       Single    Rs.1500      Available   
105       Single    Rs.1500      Available   
106       Deluxe    Rs.2800      Available   
107       Deluxe    Rs.2800      Available   
108       Deluxe    Rs.2800      Available   
109       Deluxe    Rs.2800      Available   
110       Deluxe    Rs.2800      Available   
111       Suite     Rs.5000      Available   
112       Suite     Rs.5000      Available   
113       Suite     Rs.5000      Available   
114       Suite     Rs.5000      Available   
115       Suite     Rs.5000      Available   
                HOTEL MANAGEMENT SYSTEM                
1. Show All Rooms
2. Show Available Rooms
3. Guest Check-In
4. Guest Check-Out (Generate Bill)
5. View Currently Checked-In Guests
6. Booking History
7. 

Enter your choice (1-8):  2


Room No   Type      Rate/Night  
-------------------------------------------------------
101       Single    Rs.1500      
102       Single    Rs.1500      
103       Single    Rs.1500      
104       Single    Rs.1500      
105       Single    Rs.1500      
106       Deluxe    Rs.2800      
107       Deluxe    Rs.2800      
108       Deluxe    Rs.2800      
109       Deluxe    Rs.2800      
110       Deluxe    Rs.2800      
111       Suite     Rs.5000      
112       Suite     Rs.5000      
113       Suite     Rs.5000      
114       Suite     Rs.5000      
115       Suite     Rs.5000      
                HOTEL MANAGEMENT SYSTEM                
1. Show All Rooms
2. Show Available Rooms
3. Guest Check-In
4. Guest Check-Out (Generate Bill)
5. View Currently Checked-In Guests
6. Booking History
7. Revenue / Summary Report
8. Exit
-------------------------------------------------------


...

Invalid Guest ID or guest already checked out.
                HOTEL MANAGEMENT SYSTEM                
1. Show All Rooms
2. Show Available Rooms
3. Guest Check-In
4. Guest Check-Out (Generate Bill)
5. View Currently Checked-In Guests
6. Booking History
7. Revenue / Summary Report
8. Exit
-------------------------------------------------------


Enter your choice (1-8):  6


BOOKING HISTORY (All Checkouts)
-------------------------------------------------------
No completed bookings yet.
                HOTEL MANAGEMENT SYSTEM                
1. Show All Rooms
2. Show Available Rooms
3. Guest Check-In
4. Guest Check-Out (Generate Bill)
5. View Currently Checked-In Guests
6. Booking History
7. Revenue / Summary Report
8. Exit
-------------------------------------------------------


Enter your choice (1-8):  8


Thank you for using the Hotel Management System. Goodbye!


In [4]:
# cafe management 
import json
import os
from datetime import datetime

DATA_FILE = "cafe_data.json"

# Menu: item -> price
MENU = {
    "Tea": 20,
    "Coffee": 40,
    "Cold Coffee": 60,
    "Sandwich": 80,
    "Burger": 100,
    "Pasta": 120,
    "French Fries": 70,
    "Pizza Slice": 90,
    "Cold Drink": 40,
    "Ice Cream": 50
}

GST_RATE = 0.05  # 5% GST on food bills


def load_data():
    """Load cafe data (tables, orders, sales history) or initialize fresh."""
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE, "r") as f:
            return json.load(f)

    # Initialize 8 tables
    tables = {str(i): {"occupied": False, "order_id": None} for i in range(1, 9)}

    data = {
        "tables": tables,
        "orders": {},        # order_id -> order info (active)
        "sales_history": [],  # completed bills
        "next_order_id": 1
    }
    save_data(data)
    return data


def save_data(data):
    with open(DATA_FILE, "w") as f:
        json.dump(data, f, indent=4)


data = load_data()
print("Cafe data loaded successfully!")
print(f"Total tables: {len(data['tables'])}")


def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def divider(char="-", length=55):
    print(char * length)


def show_menu():
    divider("=")
    print("CAFE MENU".center(55))
    divider("=")
    print(f"{'Item':<20}{'Price':<10}")
    divider()
    for item, price in MENU.items():
        print(f"{item:<20}Rs.{price:<10}")
    divider("=")


def show_tables(data):
    divider("=")
    print(f"{'Table No':<12}{'Status':<12}")
    divider()
    for t_no, info in sorted(data["tables"].items(), key=lambda x: int(x[0])):
        status = "Occupied" if info["occupied"] else "Available"
        print(f"{t_no:<12}{status:<12}")
    divider("=")


def new_order(data):
    divider("=")
    print("NEW ORDER / TAKE ORDER")
    divider()

    show_tables(data)
    table_no = input("Enter table number: ").strip()

    if table_no not in data["tables"]:
        print("Invalid table number.")
        return
    if data["tables"][table_no]["occupied"]:
        print("Table already occupied. Choose another or add items to existing order.")
        return

    customer_name = input("Customer name: ").strip()

    order_id = str(data["next_order_id"])
    data["next_order_id"] += 1

    items_ordered = {}
    show_menu()

    while True:
        item = input("Enter item name (or 'done' to finish): ").strip().title()
        if item.lower() == "done":
            break
        if item not in MENU:
            print("Item not found in menu. Try again.")
            continue
        try:
            qty = int(input(f"Quantity for {item}: ").strip())
            if qty <= 0:
                print("Quantity must be positive.")
                continue
        except ValueError:
            print("Enter a valid number.")
            continue

        items_ordered[item] = items_ordered.get(item, 0) + qty
        print(f"Added: {qty} x {item}")

    if not items_ordered:
        print("No items ordered. Order cancelled.")
        return

    data["orders"][order_id] = {
        "table_no": table_no,
        "customer_name": customer_name,
        "items": items_ordered,
        "placed_at": now_str(),
        "status": "active"
    }

    data["tables"][table_no]["occupied"] = True
    data["tables"][table_no]["order_id"] = order_id

    save_data(data)

    print(f"\nOrder placed successfully! Order ID: {order_id}")
    print(f"Table {table_no} is now occupied.")


def add_items_to_order(data):
    divider("=")
    print("ADD ITEMS TO EXISTING ORDER")
    divider()

    order_id = input("Enter Order ID: ").strip()
    if order_id not in data["orders"] or data["orders"][order_id]["status"] != "active":
        print("Invalid or already billed Order ID.")
        return

    order = data["orders"][order_id]
    show_menu()

    while True:
        item = input("Enter item name (or 'done' to finish): ").strip().title()
        if item.lower() == "done":
            break
        if item not in MENU:
            print("Item not found in menu. Try again.")
            continue
        try:
            qty = int(input(f"Quantity for {item}: ").strip())
            if qty <= 0:
                print("Quantity must be positive.")
                continue
        except ValueError:
            print("Enter a valid number.")
            continue

        order["items"][item] = order["items"].get(item, 0) + qty
        print(f"Added: {qty} x {item}")

    save_data(data)
    print("Order updated successfully!")


def calculate_bill(order):
    subtotal = 0
    line_items = []
    for item, qty in order["items"].items():
        price = MENU[item]
        line_total = price * qty
        subtotal += line_total
        line_items.append((item, qty, price, line_total))

    gst = round(subtotal * GST_RATE, 2)
    total = round(subtotal + gst, 2)
    return line_items, subtotal, gst, total


def generate_bill(data):
    divider("=")
    print("GENERATE BILL / CHECKOUT")
    divider()

    order_id = input("Enter Order ID: ").strip()
    if order_id not in data["orders"] or data["orders"][order_id]["status"] != "active":
        print("Invalid or already billed Order ID.")
        return

    order = data["orders"][order_id]
    line_items, subtotal, gst, total = calculate_bill(order)

    divider("=")
    print("CAFE INVOICE".center(55))
    divider("=")
    print(f"Order ID     : {order_id}")
    print(f"Table No     : {order['table_no']}")
    print(f"Customer     : {order['customer_name']}")
    print(f"Placed At    : {order['placed_at']}")
    divider()
    print(f"{'Item':<18}{'Qty':<6}{'Price':<10}{'Total':<10}")
    divider()
    for item, qty, price, line_total in line_items:
        print(f"{item:<18}{qty:<6}Rs.{price:<8}Rs.{line_total:<8}")
    divider()
    print(f"Subtotal     : Rs. {subtotal:.2f}")
    print(f"GST (5%)     : Rs. {gst:.2f}")
    divider()
    print(f"TOTAL AMOUNT : Rs. {total:.2f}")
    divider("=")

    bill_record = {
        "order_id": order_id,
        "table_no": order["table_no"],
        "customer_name": order["customer_name"],
        "items": order["items"],
        "subtotal": subtotal,
        "gst": gst,
        "total": total,
        "billed_at": now_str()
    }
    data["sales_history"].append(bill_record)

    # Free up the table
    table_no = order["table_no"]
    data["tables"][table_no]["occupied"] = False
    data["tables"][table_no]["order_id"] = None
    order["status"] = "billed"

    save_data(data)
    print("\nPayment complete. Table is now available.")


def active_orders(data):
    divider("=")
    print("ACTIVE ORDERS")
    divider()
    active = {oid: o for oid, o in data["orders"].items() if o["status"] == "active"}
    if not active:
        print("No active orders right now.")
    else:
        print(f"{'Order ID':<10}{'Table':<8}{'Customer':<18}{'Items':<8}")
        divider()
        for oid, o in active.items():
            item_count = sum(o["items"].values())
            print(f"{oid:<10}{o['table_no']:<8}{o['customer_name']:<18}{item_count:<8}")
    divider("=")


def sales_report(data):
    total_sales = sum(b["total"] for b in data["sales_history"])
    total_orders = len(data["sales_history"])

    item_count = {}
    for b in data["sales_history"]:
        for item, qty in b["items"].items():
            item_count[item] = item_count.get(item, 0) + qty

    divider("=")
    print("CAFE SALES REPORT")
    divider()
    print(f"Total Orders Billed   : {total_orders}")
    print(f"Total Revenue         : Rs. {total_sales:.2f}")
    divider()
    if item_count:
        print("Item-wise Sales:")
        for item, qty in sorted(item_count.items(), key=lambda x: -x[1]):
            print(f"  {item:<20}: {qty} sold")
    divider("=")


def main_menu():
    global data
    while True:
        divider("=")
        print("CAFE MANAGEMENT SYSTEM".center(55))
        divider("=")
        print("1. Show Menu")
        print("2. Show Tables Status")
        print("3. Take New Order")
        print("4. Add Items to Existing Order")
        print("5. View Active Orders")
        print("6. Generate Bill / Checkout")
        print("7. Sales Report")
        print("8. Exit")
        divider()

        choice = input("Enter your choice (1-8): ").strip()

        if choice == "1":
            show_menu()
        elif choice == "2":
            show_tables(data)
        elif choice == "3":
            new_order(data)
        elif choice == "4":
            add_items_to_order(data)
        elif choice == "5":
            active_orders(data)
        elif choice == "6":
            generate_bill(data)
        elif choice == "7":
            sales_report(data)
        elif choice == "8":
            print("Thank you! Cafe Management System closed.")
            break
        else:
            print("Invalid choice, please try again.")


# Run the system
main_menu()

Cafe data loaded successfully!
Total tables: 8
                 CAFE MANAGEMENT SYSTEM                
1. Show Menu
2. Show Tables Status
3. Take New Order
4. Add Items to Existing Order
5. View Active Orders
6. Generate Bill / Checkout
7. Sales Report
8. Exit
-------------------------------------------------------


Enter your choice (1-8):  1


                       CAFE MENU                       
Item                Price     
-------------------------------------------------------
Tea                 Rs.20        
Coffee              Rs.40        
Cold Coffee         Rs.60        
Sandwich            Rs.80        
Burger              Rs.100       
Pasta               Rs.120       
French Fries        Rs.70        
Pizza Slice         Rs.90        
Cold Drink          Rs.40        
Ice Cream           Rs.50        
                 CAFE MANAGEMENT SYSTEM                
1. Show Menu
2. Show Tables Status
3. Take New Order
4. Add Items to Existing Order
5. View Active Orders
6. Generate Bill / Checkout
7. Sales Report
8. Exit
-------------------------------------------------------


Enter your choice (1-8):  2


Table No    Status      
-------------------------------------------------------
1           Available   
2           Available   
3           Available   
4           Available   
5           Available   
6           Available   
7           Available   
8           Available   
                 CAFE MANAGEMENT SYSTEM                
1. Show Menu
2. Show Tables Status
3. Take New Order
4. Add Items to Existing Order
5. View Active Orders
6. Generate Bill / Checkout
7. Sales Report
8. Exit
-------------------------------------------------------


...

                      CAFE INVOICE                     
Order ID     : 1
Table No     : 3
Customer     : umar
Placed At    : 2026-07-02 00:25:42
-------------------------------------------------------
Item              Qty   Price     Total     
-------------------------------------------------------
Tea               3     Rs.20      Rs.60      
Pasta             4     Rs.120     Rs.480     
-------------------------------------------------------
Subtotal     : Rs. 540.00
GST (5%)     : Rs. 27.00
-------------------------------------------------------
TOTAL AMOUNT : Rs. 567.00

Payment complete. Table is now available.
                 CAFE MANAGEMENT SYSTEM                
1. Show Menu
2. Show Tables Status
3. Take New Order
4. Add Items to Existing Order
5. View Active Orders
6. Generate Bill / Checkout
7. Sales Report
8. Exit
-------------------------------------------------------


Enter your choice (1-8):  7


CAFE SALES REPORT
-------------------------------------------------------
Total Orders Billed   : 1
Total Revenue         : Rs. 567.00
-------------------------------------------------------
Item-wise Sales:
  Pasta               : 4 sold
  Tea                 : 3 sold
                 CAFE MANAGEMENT SYSTEM                
1. Show Menu
2. Show Tables Status
3. Take New Order
4. Add Items to Existing Order
5. View Active Orders
6. Generate Bill / Checkout
7. Sales Report
8. Exit
-------------------------------------------------------


Enter your choice (1-8):  8


Thank you! Cafe Management System closed.
